# Taxila: buffered spatial benchmark extension

## tl;dr
Execution results appear below after a complete run. This notebook compares seven model families using buffered development folds, then evaluates the previously used outer stripe. It is a **retrospective sensitivity experiment**, not an untouched confirmatory test. The response is WorldCover-derived land cover, never monument condition.

## Context & Methods
Three contiguous inner validation bands exclude adjacent block rows from training. All seven families receive three configurations and the same folds. The development mean of fixed-six-class macro-F1 selects the primary model. Paired spatial-block intervals, development-only temperature calibration, feature controls and three-seed sensitivity are reported. Early stopping is disabled to avoid hidden random validation splits.

### Key Assumptions
Samples are spatially correlated and class-stratified; overall agreement is not an area-weighted accuracy estimate. The outer stripe has been examined in earlier research. Results do not establish universal state-of-the-art performance.

### 1. Install the recorded environment
A CPU runtime is sufficient. The notebook embeds only the project feature table and the auditable experiment script, so no GitHub token or Drive mount is required.

In [1]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','numpy==2.3.5','pandas==2.2.3','scipy==1.17.0','scikit-learn==1.8.0','xgboost==3.4.1','catboost==1.2.10'])

0

## Data
### 2. Restore and verify the frozen feature table
The embedded compressed bytes are checked against the repository SHA-256 before modelling. Source: `data/Taxila_CHIP_Frozen_Evidence_Data/13_tables/baseline_reproduction/proxy_model_feature_table.csv`.

In [2]:
from pathlib import Path
import base64,gzip,hashlib
work=Path('/content/taxila_spatial_extension') if Path('/content').exists() else Path.cwd()/'taxila_spatial_extension'
work.mkdir(exist_ok=True)
data=work/'proxy_model_feature_table.csv'
payload='...'
raw=gzip.decompress(base64.b64decode(payload))
assert hashlib.sha256(raw).hexdigest()=='3188160a144c5020e7d21bb50a195e07a686d941134ade35a57232fe41ddc1b1'
data.write_bytes(raw)
print('Verified input:',len(raw),'bytes')

Verified input: 2807661 bytes


### 3. Inspect the executable method
The following source is identical to `scripts/run_spatial_extension.py`. It writes split and protocol records before evaluating any outer-test predictions.

In [3]:
source='"""Retrospective, spatially buffered land-cover proxy benchmark; never condition validation."""\nfrom __future__ import annotations\nimport os\nfor key in (\'OMP_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'MKL_NUM_THREADS\'):\n    os.environ.setdefault(key, \'2\')\nimport argparse, hashlib, json, platform, time, warnings\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom importlib.metadata import version\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import minimize_scalar\nfrom scipy.special import softmax\nfrom scipy.spatial import cKDTree\nfrom sklearn.base import clone\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.neural_network import MLPClassifier\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.metrics import f1_score, accuracy_score, log_loss, confusion_matrix\nfrom sklearn.model_selection import ParameterGrid\nfrom xgboost import XGBClassifier\nfrom catboost import CatBoostClassifier\n\nFEATURES = [\'blue\',\'green\',\'red\',\'nir08\',\'swir16\',\'swir22\',\'NDVI\',\'NDBI\',\'MNDWI\',\'BSI\']\nLABELS = np.arange(6)\nSEED = 311\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef metrics(y, p):\n    pred = p.argmax(1)\n    cm = confusion_matrix(y, pred, labels=LABELS)\n    recall = np.divide(cm.diagonal(), cm.sum(1), out=np.zeros(6), where=cm.sum(1)>0)\n    conf = p.max(1)\n    bins = np.minimum((conf*10).astype(int),9)\n    ece = sum(np.mean(bins==b)*abs(np.mean(pred[bins==b]==y[bins==b])-np.mean(conf[bins==b])) for b in range(10) if (bins==b).any())\n    return dict(macro_f1=float(f1_score(y,pred,labels=LABELS,average=\'macro\',zero_division=0)),\n                overall_agreement=float(accuracy_score(y,pred)), balanced_accuracy_fixed6=float(recall.mean()),\n                log_loss=float(log_loss(y,p,labels=LABELS)), brier=float(np.mean(np.sum((p-np.eye(6)[y])**2,axis=1))),ece=float(ece))\n\ndef candidates(seed):\n    return {\n        \'Logistic\': (make_pipeline(StandardScaler(), LogisticRegression(max_iter=1500)), {\'logisticregression__C\':[0.1,1,10]}),\n        \'RandomForest\': (RandomForestClassifier(n_estimators=300, random_state=seed,n_jobs=2,class_weight=\'balanced_subsample\'), {\'min_samples_leaf\':[1,5,12]}),\n        \'ExtraTrees\': (ExtraTreesClassifier(n_estimators=300,random_state=seed,n_jobs=2,class_weight=\'balanced\'), {\'min_samples_leaf\':[1,3,8]}),\n        \'HistGradientBoosting\': (HistGradientBoostingClassifier(max_iter=250,early_stopping=False,random_state=seed,l2_regularization=1), {\'max_leaf_nodes\':[15,31,63]}),\n        \'MLP\': (make_pipeline(StandardScaler(),MLPClassifier(max_iter=400,early_stopping=False,random_state=seed,hidden_layer_sizes=(64,32))), {\'mlpclassifier__alpha\':[0.0001,0.01,0.1]}),\n        \'XGBoost\': (XGBClassifier(n_estimators=300,learning_rate=0.05,tree_method=\'hist\',n_jobs=2,random_state=seed,objective=\'multi:softprob\',num_class=6,subsample=0.8,colsample_bytree=0.8), {\'max_depth\':[3,5,7]}),\n        \'CatBoost\': (CatBoostClassifier(iterations=300,learning_rate=0.05,loss_function=\'MultiClass\',thread_count=2,random_seed=seed,verbose=False,allow_writing_files=False), {\'depth\':[4,6,8]})\n    }\n\ndef inner_folds(frame):\n    """Contiguous north/central/south validation bands, with one block-row exclusion."""\n    rr = (frame.spatial_block.to_numpy()//10)\n    folds=[]\n    for lo,hi in [(0,2),(3,6),(7,9)]:\n        va=np.flatnonzero((rr>=lo)&(rr<=hi))\n        tr=np.flatnonzero((rr<lo-1)|(rr>hi+1))\n        assert len(tr) and len(va)\n        assert np.min(abs(np.unique(rr[tr])[:,None]-np.unique(rr[va])[None,:]))>=2\n        assert frame.proxy_class.iloc[tr].nunique() >= 2\n        folds.append((tr,va))\n    return folds\n\ndef probabilities(model, X):\n    raw=np.asarray(model.predict_proba(X))\n    p=np.zeros((len(X),6))\n    # Spatial folds may lack the geographically concentrated water class.\n    # Never create synthetic labels or silently drop that class from scoring.\n    if raw.shape[1]==6:\n        return raw\n    p[:,np.asarray(model.classes_,dtype=int)]=raw\n    return p\n\ndef fit_search(estimator,grid,X,y,folds):\n    records=[]\n    for params in ParameterGrid(grid):\n        scores=[]\n        for tr,va in folds:\n            model=clone(estimator).set_params(**params).fit(X[tr],y[tr])\n            scores.append(f1_score(y[va],np.asarray(model.predict(X[va])).ravel(),labels=LABELS,average=\'macro\',zero_division=0))\n        records.append(dict(parameters=params,fold_scores=scores,mean=float(np.mean(scores))))\n    best=max(records,key=lambda x:x[\'mean\'])\n    model=clone(estimator).set_params(**best[\'parameters\']).fit(X,y)\n    return model,best,records\n\ndef block_draw_scores(y,p,groups,draws=2000):\n    blocks=np.unique(groups)\n    cms=np.stack([confusion_matrix(y[groups==g],p[groups==g].argmax(1),labels=LABELS) for g in blocks])\n    multiplicity=np.random.default_rng(SEED).multinomial(len(blocks),np.ones(len(blocks))/len(blocks),size=draws)\n    sums=np.einsum(\'db,bij->dij\',multiplicity,cms)\n    den=sums.sum(1)+sums.sum(2)\n    f1=np.divide(2*np.diagonal(sums,axis1=1,axis2=2),den,out=np.zeros_like(den,dtype=float),where=den>0).mean(1)\n    return f1\n\ndef run(data,out):\n    started=time.time(); out=Path(out); out.mkdir(parents=True,exist_ok=False)\n    frame=pd.read_csv(data).drop(columns=[\'partition\'],errors=\'ignore\')\n    assert len(frame)==20749 and not frame.duplicated([\'row\',\'col\']).any()\n    assert np.isfinite(frame[FEATURES].to_numpy()).all()\n    assert set(frame.proxy_class)==set(range(1,7))\n    block_col=frame.spatial_block%10\n    test=block_col.isin([3,4]); buffer=block_col.isin([2,5]); dev=~(test|buffer)\n    df=frame.loc[dev].reset_index(drop=True); tf=frame.loc[test].reset_index(drop=True)\n    folds=inner_folds(df)\n    split=frame[[\'row\',\'col\',\'spatial_block\',\'proxy_class\']].copy()\n    split[\'role\']=np.where(test,\'test\',np.where(buffer,\'buffer\',\'development\'))\n    split.to_csv(out/\'split_manifest.csv\',index=False)\n    min_distance=float(cKDTree(df[[\'row\',\'col\']].to_numpy()*30).query(tf[[\'row\',\'col\']].to_numpy()*30)[0].min())\n    protocol=dict(schema=1,created_utc=datetime.now(timezone.utc).isoformat(),source_sha256=digest(data),n_samples=len(frame),\n                  seed=SEED,features=FEATURES,outer_test_columns=[3,4],outer_buffer_columns=[2,5],inner_validation_rows=[[0,2],[3,6],[7,9]],\n                  inner_buffer_rows=1,labels=list(range(6)),selection=\'largest mean inner buffered macro-F1; lexicographic model tie-break\',\n                  bootstrap_draws=2000,development_n=len(df),test_n=len(tf),buffer_n=int(buffer.sum()),minimum_test_train_distance_m=min_distance,\n                  limitation=\'Retrospective extension on an already examined outer stripe; not an untouched confirmatory test or independent condition validation.\',\n                  inner_training_class_counts=[df.proxy_class.iloc[tr].value_counts().sort_index().to_dict() for tr,va in folds],\n                  early_stopping=False,packages={p:version(p) for p in [\'numpy\',\'pandas\',\'scipy\',\'scikit-learn\',\'xgboost\',\'catboost\']},python=platform.python_version(),platform=platform.platform())\n    (out/\'protocol.json\').write_text(json.dumps(protocol,indent=2))\n    X=df[FEATURES].to_numpy(); y=df.proxy_class.to_numpy()-1\n    Xt=tf[FEATURES].to_numpy(); yt=tf.proxy_class.to_numpy()-1; groups=tf.spatial_block.to_numpy()\n    models={}; selections=[]; searches={}\n    # All selection completes before any outer-test prediction is made.\n    for name,(est,grid) in candidates(SEED).items():\n        print(\'Tuning\',name,flush=True); t=time.time()\n        model,best,history=fit_search(est,grid,X,y,folds)\n        models[name]=model; searches[name]=history\n        selections.append(dict(model=name,inner_macro_f1=best[\'mean\'],parameters=json.dumps(best[\'parameters\']),fit_seconds=time.time()-t))\n        pd.DataFrame(selections).to_csv(out/\'selection_progress.csv\',index=False)\n    sel=pd.DataFrame(selections).sort_values([\'inner_macro_f1\',\'model\'],ascending=[False,True])\n    primary=sel.iloc[0].model; sel[\'primary_selected\']=sel.model.eq(primary)\n    sel.to_csv(out/\'model_selection.csv\',index=False)\n    (out/\'inner_search.json\').write_text(json.dumps(searches,indent=2))\n    print(\'Development-selected model:\',primary,flush=True)\n    predictions=tf[[\'row\',\'col\',\'spatial_block\',\'proxy_class\']].copy(); rows=[]; bootstrap={}; probs={}\n    for name,model in models.items():\n        p=probabilities(model,Xt); probs[name]=p\n        score_draws=block_draw_scores(yt,p,groups); bootstrap[name]=score_draws\n        row=dict(model=name,primary_selected=name==primary,**metrics(yt,p),f1_low=float(np.quantile(score_draws,.025)),f1_high=float(np.quantile(score_draws,.975)))\n        rows.append(row)\n        predictions[name+\'_prediction\']=p.argmax(1)+1\n        for k in range(6): predictions[f\'{name}_p{k+1}\']=p[:,k]\n    pd.DataFrame(rows).to_csv(out/\'model_comparison.csv\',index=False)\n    predictions.to_csv(out/\'test_predictions.csv\',index=False)\n    paired=[]\n    for name in models:\n        if name==primary: continue\n        delta=bootstrap[primary]-bootstrap[name]\n        paired.append(dict(primary=primary,comparator=name,delta_macro_f1=metrics(yt,probs[primary])[\'macro_f1\']-metrics(yt,probs[name])[\'macro_f1\'],ci_low=np.quantile(delta,.025),ci_high=np.quantile(delta,.975)))\n    pd.DataFrame(paired).to_csv(out/\'paired_block_comparisons.csv\',index=False)\n    # One scalar temperature is learned only from development out-of-fold predictions.\n    oof=np.full((len(y),6),np.nan)\n    for tr,va in folds:\n        model=clone(models[primary]).fit(X[tr],y[tr]); oof[va]=probabilities(model,X[va])\n    assert np.isfinite(oof).all()\n    opt=minimize_scalar(lambda t:log_loss(y,softmax(np.log(np.clip(oof,1e-12,1))/t,axis=1),labels=LABELS),bounds=(.25,4),method=\'bounded\')\n    temp=float(opt.x); cal=softmax(np.log(np.clip(probs[primary],1e-12,1))/temp,axis=1)\n    pd.DataFrame([dict(variant=\'raw\',temperature=1.,**metrics(yt,probs[primary])),dict(variant=\'OOF-temperature\',temperature=temp,**metrics(yt,cal))]).to_csv(out/\'calibration_comparison.csv\',index=False)\n    diagnostic=[]\n    # Predeclared fixed-capacity ablations; none changes primary selection.\n    for label,cols in [(\'bands-only\',FEATURES[:6]),(\'indices-only\',FEATURES[6:]),(\'location-only\',[\'row\',\'col\'])]:\n        model=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=1500)).fit(df[cols],y)\n        diagnostic.append(dict(experiment=label,**metrics(yt,model.predict_proba(tf[cols]))))\n    null=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=1500)).fit(X,np.random.default_rng(SEED).permutation(y))\n    diagnostic.append(dict(experiment=\'shuffled-development-labels\',**metrics(yt,null.predict_proba(Xt))))\n    dummy=DummyClassifier(strategy=\'prior\').fit(X,y)\n    diagnostic.append(dict(experiment=\'class-prior\',**metrics(yt,dummy.predict_proba(Xt))))\n    pd.DataFrame(diagnostic).to_csv(out/\'diagnostic_controls.csv\',index=False)\n    seed_rows=[dict(seed=SEED,**metrics(yt,probs[primary]))]\n    primary_params=json.loads(sel.loc[sel.model.eq(primary),\'parameters\'].iloc[0])\n    for seed in [733,2026]:\n        est=candidates(seed)[primary][0].set_params(**primary_params).fit(X,y)\n        seed_rows.append(dict(seed=seed,**metrics(yt,est.predict_proba(Xt))))\n    pd.DataFrame(seed_rows).to_csv(out/\'seed_sensitivity.csv\',index=False)\n    report=dict(status=\'PASS\',primary=primary,source_sha256=digest(data),test_n=len(yt),development_n=len(y),\n                test_blocks=int(np.unique(groups).size),minimum_distance_m=min_distance,\n                primary_metrics=next(row for row in rows if row[\'model\']==primary),temperature=temp,duration_seconds=time.time()-started,\n                scope=\'Retrospective spatial land-cover proxy benchmark; no claim of heritage-condition accuracy or universal SOTA.\')\n    (out/\'summary.json\').write_text(json.dumps(report,indent=2))\n    pd.DataFrame([dict(file=p.name,sha256=digest(p),bytes=p.stat().st_size) for p in sorted(out.iterdir()) if p.is_file()]).to_csv(out/\'manifest.csv\',index=False)\n    print(json.dumps(report,indent=2),flush=True)\n    return report\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(); parser.add_argument(\'--data\',type=Path,required=True); parser.add_argument(\'--output\',type=Path,required=True)\n    args=parser.parse_args(); run(args.data,args.output)\n'
script=work/'run_spatial_extension.py'
script.write_text(source)
assert hashlib.sha256(script.read_bytes()).hexdigest()=='d2c83eab33fc08af6780e16f7425f2b5243666bf65b4e60616cadc65600e52c7'

## Results
### 4. Run all comparisons
The output directory is immutable per run. A failed assertion stops execution; no results are silently substituted.

In [4]:
from datetime import datetime,timezone
run_dir=work/('run_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
subprocess.check_call([sys.executable,'-u',str(script),'--data',str(data),'--output',str(run_dir)])

0

### 5. Read results and paired uncertainty

In [5]:
import json,pandas as pd
summary=json.loads((run_dir/'summary.json').read_text())
assert summary['status']=='PASS'
print(json.dumps(summary,indent=2))
display(pd.read_csv(run_dir/'model_comparison.csv').round(4))
display(pd.read_csv(run_dir/'paired_block_comparisons.csv').round(4))
display(pd.read_csv(run_dir/'calibration_comparison.csv').round(4))
display(pd.read_csv(run_dir/'diagnostic_controls.csv').round(4))
display(pd.read_csv(run_dir/'seed_sensitivity.csv').round(4))

{
  "status": "PASS",
  "primary": "RandomForest",
  "source_sha256": "3188160a144c5020e7d21bb50a195e07a686d941134ade35a57232fe41ddc1b1",
  "test_n": 4188,
  "development_n": 12094,
  "test_blocks": 20,
  "minimum_distance_m": 2040.0,
  "primary_metrics": {
    "model": "RandomForest",
    "primary_selected": true,
    "macro_f1": 0.8214328773275001,
    "overall_agreement": 0.8127984718242598,
    "balanced_accuracy_fixed6": 0.8018376256530257,
    "log_loss": 0.49768663809929314,
    "brier": 0.2689232267553529,
    "ece": 0.014311113874950366,
    "f1_low": 0.6910988824552918,
    "f1_high": 0.8351782871537495
  },
  "temperature": 1.6582819649510478,
  "duration_seconds": 378.00360083580017,
  "scope": "Retrospective spatial land-cover proxy benchmark; no claim of heritage-condition accuracy or universal SOTA."
}


,model,primary_selected,macro_f1,overall_agreement,balanced_accuracy_fixed6,log_loss,brier,ece,f1_low,f1_high
0,Logistic,False,0.8069,0.8032,0.7846,0.5519,0.2912,0.0498,0.6800,0.8219
1,RandomForest,True,0.8214,0.8128,0.8018,0.4977,0.2689,0.0143,0.6911,0.8352
2,ExtraTrees,False,0.8264,0.8161,0.8080,0.4862,0.2668,0.0295,0.6937,0.8405
3,HistGradientBoosting,False,0.8463,0.8348,0.8342,0.5051,0.2518,0.0514,0.7039,0.8573
4,MLP,False,0.8495,0.8546,0.8283,0.4265,0.2179,0.0203,0.7229,0.8885
5,XGBoost,False,0.8269,0.8231,0.8086,0.4734,0.2555,0.0167,0.6991,0.8446
6,CatBoost,False,0.7924,0.8157,0.7665,0.4854,0.2643,0.0167,0.6804,0.8251


,primary,comparator,delta_macro_f1,ci_low,ci_high
0,RandomForest,Logistic,0.0145,0.0028,0.0283
1,RandomForest,ExtraTrees,-0.0049,-0.0099,0.0014
2,RandomForest,HistGradientBoosting,-0.0248,-0.0663,-0.0117
3,RandomForest,MLP,-0.0281,-0.1063,0.0504
4,RandomForest,XGBoost,-0.0054,-0.0587,0.0094
5,RandomForest,CatBoost,0.0290,-0.0051,0.0990


,variant,temperature,macro_f1,overall_agreement,balanced_accuracy_fixed6,log_loss,brier,ece
0,raw,1.0000,0.8214,0.8128,0.8018,0.4977,0.2689,0.0143
1,OOF-temperature,1.6583,0.8214,0.8128,0.8018,0.5735,0.2999,0.1357


,experiment,macro_f1,overall_agreement,balanced_accuracy_fixed6,log_loss,brier,ece
0,bands-only,0.8245,0.7954,0.8215,0.5619,0.2993,0.0513
1,indices-only,0.6706,0.6566,0.6506,0.8156,0.4490,0.0337
2,location-only,0.1404,0.2290,0.1817,1.9122,0.8508,0.1467
3,shuffled-development-labels,0.0699,0.1538,0.1488,1.7117,0.8174,0.0627
4,class-prior,0.0495,0.1743,0.1667,1.7062,0.8147,0.0410


,seed,macro_f1,overall_agreement,balanced_accuracy_fixed6,log_loss,brier,ece
0,311,0.8214,0.8128,0.8018,0.4977,0.2689,0.0143
1,733,0.8214,0.8128,0.8019,0.5002,0.2698,0.0142
2,2026,0.8205,0.8116,0.8008,0.4987,0.2698,0.0133


## Takeaways
Read the development-selected model, not the highest outer-test score, as the primary comparison. Intervals describe conditional spatial resampling on 20 test blocks; they do not cover label errors, site-to-site transfer or heritage damage. A negative paired interval or worse calibration result must be retained.

### 6. Preserve execution evidence
Download includes per-sample class probabilities, split manifest, search history, calibration, controls, seed sensitivity, package versions and checksums.

In [6]:
import shutil
archive=shutil.make_archive(str(run_dir),'zip',run_dir)
print('Evidence archive:',archive)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(archive)

Evidence archive: /content/taxila_spatial_extension/run_20260906T125746Z.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>